# Monkeypox skin-lesion classification (VGG16 transfer learning)

Four classes: **Monkeypox, Measles, Chickenpox, Normal**.

Corrections over the first version of this notebook:

1. **Split first, balance only the training set.** The old notebook oversampled before splitting, so duplicate images landed in train, validation and test and inflated the scores. Validation and test now contain only untouched, unique images.
2. **VGG16's own `preprocess_input`** instead of `rescale=1/255`.
3. **Augmentation is actually used** for training (it was defined but never applied).
4. **Class order is explicit** (`classes=[...]`) and `class_indices` is printed, so per-class results are unambiguous.
5. Fixed the cell-0 `SyntaxError`, the mislabeled pie/donut charts, the `plt.asix` typo, the invalid colour code, the export path, and removed the unused second model and unused class weights.

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
CLASSES = ['Chickenpox', 'Measles', 'Monkeypox', 'Normal']   # fixed order -> indices 0..3
IMG_EXT = ('.png', '.jpg', '.jpeg', '.bmp')

## 1. Load the dataset

On Kaggle, attach *Monkeypox Skin Image Dataset*. Elsewhere, point `DATA_DIR` at a folder with one sub-folder per class.

In [ ]:
KAGGLE_DIR = '/kaggle/input/monkeypoxskinimagedataset/Monkeypox Skin Image Dataset'
DATA_DIR = KAGGLE_DIR if os.path.isdir(KAGGLE_DIR) else '.'   # edit if running locally
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

def get_data(data_dir):
    rows = []
    for cls in sorted(os.listdir(data_dir)):
        folder = os.path.join(data_dir, cls)
        if cls not in CLASSES or not os.path.isdir(folder):
            continue
        for f in sorted(os.listdir(folder)):
            if f.lower().endswith(IMG_EXT):
                rows.append({'file_path': os.path.join(folder, f), 'labels': cls})
    return pd.DataFrame(rows)

df = get_data(DATA_DIR)
assert len(df) > 0, f'No images found under {DATA_DIR}'
counts = df['labels'].value_counts()
print(counts, '\nTotal:', len(df))

## 2. Explore

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(x='labels', data=df, order=counts.index, ax=ax[0])
ax[0].set_title('Images per class'); ax[0].set_xlabel('Class'); ax[0].set_ylabel('Count')

# labels come from the same Series as the values, so slices can never be mislabeled
colors = dict(zip(CLASSES, ['#ADFF2F', '#0000FF', '#FF0000', '#FFD700']))
ax[1].pie(counts.values, labels=counts.index, autopct='%1.f%%', shadow=True,
          colors=[colors[c] for c in counts.index], explode=[0.1] + [0] * (len(counts) - 1))
ax[1].set_title('Class share')

ax[2].pie(counts.values, labels=counts.index, autopct='%1.f%%', pctdistance=0.85,
          colors=[colors[c] for c in counts.index], explode=[0.05] * len(counts))
ax[2].add_artist(plt.Circle((0, 0), 0.70, fc='white'))
ax[2].set_title('Class share (donut)')
plt.tight_layout(); plt.show()

In [ ]:
n_per_class = 4
present = [c for c in CLASSES if c in set(df['labels'])]
plt.figure(figsize=(12, len(present) * 3))
for i, cls in enumerate(present):
    paths = df.loc[df['labels'] == cls, 'file_path'].head(n_per_class)
    for j, p in enumerate(paths):
        plt.subplot(len(present), n_per_class, i * n_per_class + j + 1)
        plt.imshow(Image.open(p)); plt.axis('off'); plt.title(cls)
plt.tight_layout(); plt.show()

## 3. Split first, then balance only the training set

Stratified 80 / 10 / 10 split on the **original** images. Only `train_df` is oversampled; `val_df` and `test_df` stay untouched, so no image can appear in two splits.

In [ ]:
train_df, temp_df = train_test_split(df, train_size=0.8, stratify=df['labels'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, train_size=0.5, stratify=temp_df['labels'], random_state=SEED)

# leak check: no file may appear in more than one split
assert not (set(train_df.file_path) & set(val_df.file_path))
assert not (set(train_df.file_path) & set(test_df.file_path))
assert not (set(val_df.file_path) & set(test_df.file_path))

# oversample minority classes in the TRAINING set only (augmentation makes each copy differ)
target = train_df['labels'].value_counts().max()
train_bal = pd.concat(
    [g.sample(n=target, replace=True, random_state=SEED) for _, g in train_df.groupby('labels')]
).sample(frac=1, random_state=SEED).reset_index(drop=True)

print('train (before balance):', train_df['labels'].value_counts().to_dict())
print('train (after balance): ', train_bal['labels'].value_counts().to_dict())
print('val:', val_df['labels'].value_counts().to_dict())
print('test:', test_df['labels'].value_counts().to_dict())

## 4. Data generators

`preprocess_input` is VGG16's own preprocessing (RGB to BGR, ImageNet mean subtraction). Augmentation is applied to training images only.

In [ ]:
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20, horizontal_flip=True, shear_range=0.2, zoom_range=0.2,
    height_shift_range=0.2, width_shift_range=0.2, fill_mode='nearest')
eval_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

def flow(gen, frame, shuffle):
    return gen.flow_from_dataframe(
        frame, x_col='file_path', y_col='labels', classes=CLASSES,
        target_size=IMG_SIZE, batch_size=BATCH_SIZE, color_mode='rgb',
        class_mode='categorical', shuffle=shuffle, seed=SEED)

train_ds = flow(train_gen, train_bal, shuffle=True)
val_ds = flow(eval_gen, val_df, shuffle=False)
test_ds = flow(eval_gen, test_df, shuffle=False)   # shuffle=False keeps predictions aligned with test_ds.classes

print('class_indices:', train_ds.class_indices)
assert train_ds.class_indices == val_ds.class_indices == test_ds.class_indices

## 5. Model: frozen VGG16 + a trainable softmax head

In [ ]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,))
base_model.trainable = False

x = Flatten()(base_model.output)
output = Dense(len(CLASSES), activation='softmax')(x)
monkeypox = Model(base_model.input, output)

monkeypox.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
monkeypox.summary()

## 6. Train

In [ ]:
history = monkeypox.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, k, t in zip(ax, ['loss', 'accuracy'], ['Loss', 'Accuracy']):
    a.plot(history.history[k], label='train')
    a.plot(history.history['val_' + k], label='validation')
    a.set_title(t); a.set_xlabel('Epoch'); a.set_ylabel(t); a.legend()
plt.tight_layout(); plt.show()

## 7. Evaluate

The test set is small, so treat the accuracy as an estimate (one extra mistake moves it by roughly 1 point).

In [ ]:
for name, ds in [('Train', train_ds), ('Validation', val_ds), ('Test', test_ds)]:
    loss, acc = monkeypox.evaluate(ds, verbose=0)
    print(f'{name:<10} loss={loss:.4f}  accuracy={acc:.4f}')

In [ ]:
pred = monkeypox.predict(test_ds)
y_pred = np.argmax(pred, axis=1)
y_true = test_ds.classes

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Test confusion matrix'); plt.show()

print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

## 8. Save and download

In [ ]:
from IPython.display import FileLink

model_path = os.path.join(OUT_DIR, 'monkeypox.keras')
monkeypox.save(model_path)
print('saved to', model_path)
FileLink(model_path)   # on Kaggle: shows a download link